In [ ]:
import numpy as np
from datetime import datetime, timedelta

FILE_NAME = "/content/WhatsApp Chat with Sec A Msec (notes).txt"

with open(FILE_NAME, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines in file:", len(lines))

def is_date_line(line):
    if len(line) < 8:
        return False

    first_part = line[:8]

    return (
        first_part[2] == "/" and
        first_part[5] == "/" and
        first_part[0:2].isdigit() and
        first_part[3:5].isdigit() and
        first_part[6:8].isdigit()
    )


messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

current_message = None

for raw_line in lines:

    line = raw_line.strip()

    if line == "":
        continue

    # Handle multi-line messages
    if not is_date_line(line):
        if current_message is not None:
            current_message["text"] += " " + line
        continue

    # Save previous message
    if current_message is not None:
        messages.append(current_message)

    current_message = None

    # Split timestamp and remaining content
    parts = line.split(" - ", 1);

    if len(parts) != 2:
        continue

    timestamp = parts[0]
    remaining = parts[1]

    # Split sender and message
    sender_parts = remaining.split(": ", 1)

    if len(sender_parts) != 2:
        system_messages += 1
        continue

    sender = sender_parts[0]
    text = sender_parts[1]

    current_message = {
        "timestamp": timestamp,
        "sender": sender,
        "text": text
    }

# Add final message
if current_message is not None:
    messages.append(current_message)


# Separate media and deleted messages
real_messages = []

for msg in messages:

    text = msg["text"]

    if text == "<Media omitted>":
        media_messages += 1
        continue

    if text == "This message was deleted":
        deleted_messages += 1
        continue

    real_messages.append(msg)


messages = real_messages

print("Successfully parsed:", len(messages), "real messages")
print("System messages:", system_messages)
print("Media messages:", media_messages)
print("Deleted messages:", deleted_messages)

for msg in messages:
    # Replace narrow no-break space with regular space for consistent parsing
    clean_timestamp = msg["timestamp"].replace('\u202f', ' ')
    msg["datetime"] = datetime.strptime(
        clean_timestamp,
        "%d/%m/%y, %I:%M %p" # Changed format to handle 12-hour time and AM/PM
    )

print(messages[0])

participants = set()

message_count = {}

for msg in messages:

    person = msg["sender"]

    participants.add(person);

    if person not in message_count:
        message_count[person] = 0

    message_count[person] += 1


first_date = min(msg["datetime"] for msg in messages)
last_date = max(msg["datetime"] for msg in messages)

total_days = (last_date.date() - first_date.date()).days + 1

print("=" * 60)
print("GROUPDNA — GROUP OVERVIEW")
print("=" * 60)

print("Period       :", first_date.strftime("%d %B %Y"),
      "to",
      last_date.strftime("%d %B %Y"))

print("Total days   :", total_days)
print(
    "Total messages:", len(messages)
)
print("Participants :", len(participants))

print("\nMESSAGES PER PERSON")

ranking = sorted(
    message_count.items(),
    key=lambda x: x[1],
    reverse=True
)

for person, count in ranking:

    percentage = count / len(messages) * 100

    print(
        f"{person:<12} : {count:>4} "
        f"({percentage:5.1f}%)"
    )

day_counts = {}
hour_counts = {}

for msg in messages:

    date = msg["datetime"].date()
    hour = msg["datetime"].hour

    day_counts[date] = day_counts.get(date, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1


busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)

print("\n" + "=" * 60)
print("ACTIVITY PEAKS")
print("=" * 60)

print(
    "Busiest day :",
    busiest_day.strftime("%d %B %Y"),
    f"({day_counts[busiest_day]} messages)"
)

print(
    f"Busiest hour: {busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00 "
    f"({hour_counts[busiest_hour]} messages)"
)

day_counts = {}
hour_counts = {}

for msg in messages:

    date = msg["datetime"].date()
    hour = msg["datetime"].hour

    day_counts[date] = day_counts.get(date, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1


busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)

people = sorted(participants)

person_index = {}

for i, person in enumerate(people):
    person_index[person] = i


heatmap = np.zeros((len(people), 24), dtype=int)


for msg in messages:

    person = msg["sender"]
    hour = msg["datetime"].hour

    row = person_index[person]

    heatmap[row][hour] += 1


print("\n" + "=" * 60)
print("ACTIVITY HEATMAP")
print("=" * 60)

print("       " + " ".join(f"{h:02d}" for h in range(24)))

for i, person in enumerate(people):

    row = heatmap[i]

    maximum = np.max(row)

    output = []

    for value in row:

        if maximum == 0:
            symbol = "."
        else:

            ratio = value / maximum

            if ratio <= 0.25:
                symbol = "."
            elif ratio <= 0.50:
                symbol = "░"
            elif ratio <= 0.75:
                symbol = "▒"
            else:
                symbol = "█"

        output.append(symbol)

    print(f"{person:<10}" + " ".join(output))


people = sorted(participants)

person_index = {}

for i, person in enumerate(people):
    person_index[person] = i


heatmap = np.zeros((len(people), 24), dtype=int)


for msg in messages:

    person = msg["sender"]
    hour = msg["datetime"].hour

    row = person_index[person]

    heatmap[row][hour] += 1


print("\n" + "=" * 60)
print("ACTIVITY HEATMAP")
print("=" * 60)

print("       " + " ".join(f"{h:02d}" for h in range(24)))

for i, person in enumerate(people):

    row = heatmap[i]

    maximum = np.max(row)

    output = []

    for value in row:

        if maximum == 0:
            symbol = "."
        else:

            ratio = value / maximum

            if ratio <= 0.25:
                symbol = "."
            elif ratio <= 0.50:
                symbol = "░"
            elif ratio <= 0.75:
                symbol = "▒"
            else:
                symbol = "█"

        output.append(symbol)

    print(f"{person:<10}" + " ".join(output))

stop_words = {
    "i", "is", "the", "a", "and", "or",
    "to", "of", "in", "on", "for",
    "it", "this", "that", "you", "me",
    "my", "we", "are", "was", "be"
}

word_count = {}

punctuation = ".,!?;:'\"()[]{}<>-/\\"

for msg in messages:

    words = msg["text"].lower().split()

    for word in words:

        word = word.strip(punctuation)

        if word == "":
            continue

        if word in stop_words:
            continue

        word_count[word] = word_count.get(word, 0) + 1


top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]


print("\n" + "=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)

for word, count in top_words:

    bar_length = min(30, count // 5)

    bar = "█" * bar_length

    print(f"{word:<15} {bar} {count}")

person_messages = {}

for person in people:
    person_messages[person] = []


for msg in messages:
    person_messages[msg["sender"]].append(msg)


average_words = {}

for person in people:

    total_words = 0

    for msg in person_messages[person]:
        total_words += len(msg["text"].split())

    count = len(person_messages[person])

    if count > 0:
        average_words[person] = total_words / count
    else:
        average_words[person] = 0


print("\n" + "=" * 60)
print("AVERAGE MESSAGE LENGTH")
print("=" * 60)

for person in people:

    print(
        f"{person:<12} : "
        f"{average_words[person]:.1f} words/message"
    )

response_gaps = {}

for person in people:
    response_gaps[person] = []


for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    # Response only when another person sends the next message
    if previous["sender"] != current["sender"]:

        gap = (
            current["datetime"] -
            previous["datetime"]
        ).total_seconds()

        # Ignore very large gaps
        if gap >= 0:
            response_gaps[current["sender"]].append(gap)


average_response = {}

for person in people:

    gaps = response_gaps[person]

    if len(gaps) > 0:
        average_response[person] = (
            sum(gaps) / len(gaps)
        )
    else:
        average_response[person] = float("inf")


print("\n" + "=" * 60)
print("RESPONSE PATTERNS")
print("=" * 60)

for person in people:

    seconds = average_response[person]

    if seconds == float("inf"):
        print(f"{person:<12} : No response data")
        continue

    minutes = seconds / 60

    if minutes < 60:
        print(
            f"{person:<12} : "
            f"{minutes:.1f} minutes"
        )
    else:
        print(
            f"{person:<12} : "
            f"{minutes / 60:.1f} hours"
        )

# Calculate silent streaks
silent_streaks = {}

for person in people:
    person_msgs = person_messages[person]

    if len(person_msgs) < 2:
        silent_streaks[person] = total_days  # Assume silent for total duration if less than 2 messages
        continue

    # Calculate gaps between consecutive messages for the person
    gaps_for_person = []
    for i in range(1, len(person_msgs)):
        gap = (person_msgs[i]["datetime"] - person_msgs[i-1]["datetime"]).total_seconds()
        gaps_for_person.append(gap)

    # Also consider the gap from the chat start to the person's first message
    first_msg_gap = (person_msgs[0]["datetime"] - first_date).total_seconds()
    if first_msg_gap > 0:
        gaps_for_person.append(first_msg_gap)

    # And the gap from the person's last message to the chat end
    last_msg_gap = (last_date - person_msgs[-1]["datetime"]).total_seconds()
    if last_msg_gap > 0:
        gaps_for_person.append(last_msg_gap)

    if gaps_for_person:
        # Convert longest gap from seconds to days
        longest_gap_seconds = max(gaps_for_person)
        silent_streaks[person] = longest_gap_seconds / (24 * 3600) # days
    else:
        silent_streaks[person] = total_days # Default if no gaps (e.g., only one message for person)

def spammer_score(person):

    msgs = person_messages[person]

    if len(msgs) == 0:
        return 0

    bursts = []
    current_burst = 0

    previous_sender = None

    for msg in msgs:

        # Count consecutive messages in original chat
        index = messages.index(msg)

        if index == 0:
            current_burst = 1
        else:

            previous = messages[index - 1]

            if previous["sender"] == person:
                current_burst += 1
            else:
                if current_burst > 0:
                    bursts.append(current_burst)

                current_burst = 1

    bursts.append(current_burst)

    return sum(bursts) / len(bursts)


def group_mom_score(person):

    caring_words = [
        "okay",
        "safe",
        "eat",
        "sleep",
        "take care",
        "are you",
        "please",
        "reminder",
        "drink water",
        "don't forget"
    ]

    score = 0

    for msg in person_messages[person]:

        text = msg["text"].lower()

        for word in caring_words:

            if word in text:
                score += 1

    return score


def night_owl_score(person):

    msgs = person_messages[person]

    if len(msgs) == 0:
        return 0

    night_count = 0

    for msg in msgs:

        hour = msg["datetime"].hour

        if hour >= 23 or hour <= 4:
            night_count += 1

    return night_count / len(msgs)


def storyteller_score(person):

    return average_words[person]


def drama_queen_score(person):

    msgs = person_messages[person]

    if len(msgs) == 0:
        return 0

    score = 0

    for msg in msgs:

        text = msg["text"]

        letters = ""

        for char in text:
            if char.isalpha():
                letters += char

        all_caps = (
            len(letters) >= 3 and
            letters.isupper()
        )

        many_exclamations = text.count("!") >= 2

        if all_caps or many_exclamations:
            score += 1

    return score / len(msgs)


def ghost_score(person):

    return silent_streaks[person] / total_days


def comedian_score(person):

    funny_words = [
        "lol",
        "lmao",
        "haha",
        "rofl",
        "lmfao"
    ]

    msgs = person_messages[person]

    if len(msgs) == 0:
        return 0

    count = 0

    for msg in msgs:

        text = msg["text"].lower()

        for word in funny_words:
            if word in text:
                count += 1

    return count / len(msgs)


def question_master_score(person):

    msgs = person_messages[person]

    if len(msgs) == 0:
        return 0

    count = 0

    for msg in msgs:

        if msg["text"].strip().endswith("?"):
            count += 1

    return count / len(msgs)

print("\n")
print("=" * 70)
print('          GROUPDNA — "HOSTEL BOIS 4EVER"')
print("          Your WhatsApp Group Chat, Decoded")
print("=" * 70)

print(
    f"{total_days} days • "
    f"{len(messages)} messages • "
    f"{len(people)} members"
)

print("=" * 70)

print("\nGROUP OVERVIEW")
print("-" * 70)

print(
    "Period:",
    first_date.strftime("%d %B %Y"),
    "to",
    last_date.strftime("%d %B %Y")
)

print("\nMESSAGES PER PERSON")

for person, count in ranking:

    percentage = count / len(messages) * 100

    bar = "█" * int(percentage / 2)

    print(
        f"{person:<10} "
        f"{bar:<20} "
        f"{count:>4} "
        f"({percentage:4.1f}%)"
    )


print("\nACTIVITY PEAKS")
print("-" * 70)

print(
    f"Busiest day : "
    f"{busiest_day.strftime('%d %B %Y')} "
    f"({day_counts[busiest_day]} messages)"
)

print(
    f"Busiest hour: "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00"
)


print("\nACTIVITY HEATMAP")
print("-" * 70)

print(
    "          " +
    " ".join(f"{h:02d}" for h in range(24))
)

for i, person in enumerate(people):

    row = heatmap[i]
    maximum = np.max(row)

    symbols = []

    for value in row:

        if maximum == 0:
            symbol = "."

        else:

            ratio = value / maximum

            if ratio <= 0.25:
                symbol = "."
            elif ratio <= 0.50:
                symbol = "░"
            elif ratio <= 0.75:
                symbol = "▒"
            else:
                symbol = "█"

        symbols.append(symbol)

    print(
        f"{person:<10}" +
        " ".join(symbols)
    )


print("\nTOP WORDS")
print("-" * 70)

for word, count in top_words[:10]:

    bar = "█" * min(30, count // 5)

    print(
        f"{word:<15} "
        f"{bar:<30} "
        f"{count}"
    )


print("\nRESPONSE PATTERNS")
print("-" * 70)

valid_response = {
    p: v for p, v in average_response.items()
    if v != float("inf")
}

if valid_response:

    fastest = min(
        valid_response,
        key=valid_response.get
    )

    slowest = max(
        valid_response,
        key=valid_response.get
    )

    print(
        "Fastest replier:",
        fastest
    )

    print(
        "Slowest replier:",
        slowest
    )


print("\nLONGEST SILENT STREAKS")
print("-" * 70)

for person in sorted(
    people,
    key=lambda x: silent_streaks[x],
    reverse=True
):

    print(
        f"{person:<10} : "
        f"{silent_streaks[person]:.1f} days"
    )

# Calculate archetypes
assigned_archetypes = {}

# Define thresholds for archetype assignment (these can be tuned)
SPAMMER_THRESHOLD = 2.0  # Average burst length
MOM_THRESHOLD = 5       # Number of caring words
NIGHT_OWL_THRESHOLD = 0.3 # Percentage of night messages
STORYTELLER_THRESHOLD = 15.0 # Average words per message
DRAMA_QUEEN_THRESHOLD = 0.1 # Percentage of dramatic messages
GHOST_THRESHOLD = 0.5   # Percentage of total days silent
COMEDIAN_THRESHOLD = 0.05 # Percentage of funny messages
QUESTION_MASTER_THRESHOLD = 0.05 # Percentage of question messages

for person in people:
    # Calculate all scores for the current person
    current_person_scores = {
        "Spammer": spammer_score(person),
        "Group Mom": group_mom_score(person),
        "Night Owl": night_owl_score(person),
        "Storyteller": storyteller_score(person),
        "Drama Queen": drama_queen_score(person),
        "Ghost": ghost_score(person),
        "Comedian": comedian_score(person),
        "Question Master": question_master_score(person)
    }

    # Initialize with a default archetype
    assigned_archetypes[person] = "Regular User"

    # Sort scores to find the highest, considering thresholds
    # Exclude Ghost for very low activity users, as other archetypes might not make sense
    if len(person_messages[person]) < 5: # Heuristic for very low activity users
        assigned_archetypes[person] = "Ghost (Low Activity)"
        continue

    # Check for specific archetypes in order of potential impact or general interest
    if current_person_scores["Spammer"] > SPAMMER_THRESHOLD:
        assigned_archetypes[person] = "Spammer"
    elif current_person_scores["Drama Queen"] > DRAMA_QUEEN_THRESHOLD:
        assigned_archetypes[person] = "Drama Queen"
    elif current_person_scores["Group Mom"] > MOM_THRESHOLD:
        assigned_archetypes[person] = "Group Mom"
    elif current_person_scores["Comedian"] > COMEDIAN_THRESHOLD:
        assigned_archetypes[person] = "Comedian"
    elif current_person_scores["Night Owl"] > NIGHT_OWL_THRESHOLD:
        assigned_archetypes[person] = "Night Owl"
    elif current_person_scores["Storyteller"] > STORYTELLER_THRESHOLD:
        assigned_archetypes[person] = "Storyteller"
    elif current_person_scores["Question Master"] > QUESTION_MASTER_THRESHOLD:
        assigned_archetypes[person] = "Question Master"
    elif current_person_scores["Ghost"] > GHOST_THRESHOLD: # Ghost is checked last as it might overlap with low activity
        assigned_archetypes[person] = "Ghost"

print("\nPERSONALITY ARCHETYPES")
print("-" * 70)

for person in people:

    print(
        f"{person:<10} → "
        f"{assigned_archetypes[person]}"
    )


print("\n" + "=" * 70)
print("       Generated by GroupDNA • Python + NumPy")
print("=" * 70)


Total lines in file: 5895
Successfully parsed: 2361 real messages
System messages: 267
Media messages: 954
Deleted messages: 255
{'timestamp': '26/03/24, 6:47\u202fpm', 'sender': 'Sajidha', 'text': 'This is for reference guys', 'datetime': datetime.datetime(2024, 3, 26, 18, 47)}
GROUPDNA — GROUP OVERVIEW
Period       : 26 March 2024 to 19 August 2026
Total days   : 877
Total messages: 2361
Participants : 71

MESSAGES PER PERSON
Sajidha      :  759 ( 32.1%)
+91 94806 25118 :  137 (  5.8%)
+91 93981 18446 :  112 (  4.7%)
+91 93417 74601 :   92 (  3.9%)
+91 63662 64470 :   85 (  3.6%)
Ananya Achalkar :   84 (  3.6%)
+91 81053 22912 :   81 (  3.4%)
+91 87925 89156 :   78 (  3.3%)
+91 86608 29278 :   75 (  3.2%)
Anu Keerthana :   70 (  3.0%)
+91 78992 93193 :   64 (  2.7%)
+91 86604 18318 :   64 (  2.7%)
+91 63018 58097 :   63 (  2.7%)
+91 95914 43954 :   47 (  2.0%)
Aayushi Sharma :   41 (  1.7%)
+91 63647 66998 :   38 (  1.6%)
+91 80736 19116 :   37 (  1.6%)
Bhoomika Msec :   35 (  1.5%)
